# Machine Learning — Market Direction Prediction — Week 3

**Aayan Mulani | Decimal Point Analytics Preparation**

This notebook builds and compares machine learning classifiers (Logistic Regression, Decision Tree, Random Forest) to predict whether the Nifty 50 index will go up or down the next day. We engineer features from price history, train on historical data, and evaluate performance on unseen data the model hasn't trained on.

## 1. Data and Feature Engineering

We download Nifty 50 price history and build four predictive features (Return, MA5, MA20, Momentum) along with a Target column — whether tomorrow's return is positive. These features become the "clues" our classifier uses to predict market direction.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

nifty = yf.download('^NSEI', start='2018-01-01', end='2025-01-01')['Close'].squeeze()
df    = pd.DataFrame({'Close': nifty})
df['Return']   = df['Close'].pct_change()
df['MA5']      = df['Close'].rolling(5).mean()
df['MA20']     = df['Close'].rolling(20).mean()
df['Momentum'] = df['Close'] / df['Close'].shift(10) - 1
df['Target']   = (df['Return'].shift(-1) > 0).astype(int)  # 1=up, 0=down tomorrow
df.dropna(inplace=True)

## 2. Train/Test Split and Logistic Regression

We split the data chronologically (no shuffling, since this is time series — shuffling would let the model "peek" at future data) and train a logistic regression classifier on the training set. We then evaluate it on the unseen test set to see how well it predicts market direction.

In [ ]:
features = ['Return', 'MA5', 'MA20', 'Momentum']
X = df[features]
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

**Interpretation:** 53.67% accuracy is only marginally above the 50% random baseline, consistent with the Efficient Market Hypothesis — past price-based features carry limited predictive power for next-day direction. The model shows recall imbalance (78% for "up" vs 23% for "down"), suggesting it leans on the market's overall upward bias in this period rather than a genuine directional signal.

## 3. Decision Tree Classifier

We train a decision tree (capped at depth 4 to avoid overfitting) on the same features and compare its accuracy against logistic regression. We also inspect feature importance to see which inputs the tree relied on most.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print(f'Decision Tree Accuracy: {accuracy_score(y_test, y_pred_tree):.4f}')
print('\nFeature Importance:')
for feat, imp in zip(features, tree.feature_importances_):
    print(f'  {feat:<12}: {imp:.4f}')